# Sổ tay 3 — Chuẩn hóa văn bản phương ngữ bằng mBART và LoRA

Notebook này là phần chính của đồ án: dùng một mô hình **encoder-decoder**
để dịch câu phương ngữ về câu tiếng Việt chuẩn, đo khoảng cách giữa bản gốc
và bản dịch, rồi fine-tune bằng LoRA nếu khoảng cách đó còn lớn.

## Mục tiêu học tập

1. Giải thích vì sao bài toán normalization dùng mô hình seq2seq, không phải encoder-only.
2. Chạy baseline mBART private trên một mẫu dialect và sinh bản chuẩn hóa.
3. **So sánh câu gốc (dialect) với câu dịch (normalized) về độ dài và độ phức tạp
   (perplexity)** — đây là thí nghiệm trung tâm của notebook.
4. Đo CER/WER giữa bản dịch và bản chuẩn vàng.
5. Fine-tune mBART bằng LoRA và so sánh với baseline trên cùng test split.
6. Phân loại lỗi thủ công và viết kết luận có bằng chứng.

## Khái niệm: text normalization là gì?

**Text normalization** là phép biến đổi một văn bản về một dạng chuẩn, giữ nguyên
nghĩa. Trong đồ án này, "dạng chuẩn" = tiếng Việt phổ thông, và input = câu viết
bằng phương ngữ (PNB/PNT2/PNT3).

Đây là bài toán **sequence-to-sequence (seq2seq)**: đọc một chuỗi dialect, sinh
một chuỗi standard. Phép biến đổi có thể là thay từ vựng, sửa hậu tố, đổi ngữ pháp
hoặc để nguyên khi câu đã chuẩn. Vì output là chuỗi tự do (không phải nhãn), ta
cần một mô hình có cả encoder (đọc input) và decoder (sinh output).

Tham khảo CS224n: mô hình encoder-decoder factorize `p(y|x)` thành
`encoder(x) → context → decoder(context, y_<t) → y_t`. Đây chính là kiến trúc
của mBART.

## mBERT hay mBART? Phân biệt quan trọng

Đồ án dùng **mBART** (`facebook/mbart-large-50` / checkpoint private
`tarudesu/mbart-large-50`), **không phải mBERT**. Hai mô hình này rất khác nhau:

| Khía cạnh | mBERT | mBART |
|---|---|---|
| Kiến trúc | encoder-only | encoder-decoder (seq2seq) |
| Hàm loss | masked LM | denoising seq2seq |
| Output | biểu diễn ẩn / nhãn | chuỗi token tự do |
| Phù hợp | phân loại, embedding | dịch, tóm tắt, **normalization** |

Vì bài toán normalization cần **sinh chuỗi**, mBERT (encoder-only) không làm được
trực tiếp. mBART có decoder nên gán được `p(standard | dialect)`. Đây là lý do
checkpoint tốt nhất trong benchmark là mBART, không phải mBERT.

> **Lưu ý giảng dạy:** Tên hai mô hình dễ nhầm. Nếu đọc "mBERT" trong tài liệu cũ,
> hãy kiểm tra kiến trúc: nếu nó dịch được chuỗi, đó là mBART.

In [ ]:
from pathlib import Path
import sys

# Detect project root whether we run from notebooks/ or project root.
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "data" / "train_240.jsonl").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run the notebook from the project root or notebooks/ directory.")
sys.path.insert(0, str(ROOT / "src"))
print(f"Project root: {ROOT}")

In [ ]:
import subprocess
import sys

INSTALL_DEPS = False
if INSTALL_DEPS:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.45,<5", "torch>=2.2,<3",
        "sentencepiece>=0.2,<1", "peft>=0.11,<1",
        "datasets>=2.19,<3", "accelerate>=0.30,<1",
        "evaluate>=0.4,<1", "sacrebleu>=2.4,<3",
    ])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from vialect_seas.data import DIALECTS, TASKS, load_jsonl
from vialect_seas.metrics import character_error_rate, word_error_rate, exact_match
from vialect_seas.normalization import (
    BASE_MODEL_ID, PRIVATE_NORMALIZER_ID,
    get_hf_token, load_seq2seq_model, generate_normalizations,
    attach_lora, make_preprocess_function,
)

sns.set_theme(style="whitegrid", context="notebook")
train = load_jsonl(ROOT / "data" / "train_240.jsonl")
test = load_jsonl(ROOT / "data" / "test_300.jsonl")
print(f"train={len(train)}  test={len(test)}")

### STUDENT TASK 0 — Kế hoạch thí nghiệm

Trước khi tải mô hình, ghi lại dự đoán của nhóm. Đây là preregistration: giúp
nhóm không tự điều chỉnh hypothesis sau khi thấy kết quả.

In [ ]:
# HINT: RQ nên nêu rõ (1) input, (2) yếu tố so sánh, (3) metric.
"""Your code here"""
TEAM_NAME = "TODO"
NORM_RQ = "TODO: ví dụ - bản dịch mBART có gần chuẩn vàng hơn câu dialect gốc không, về độ dài và perplexity?"
# Dự đoán hướng của hai phép so sánh (tăng/giảm/không đổi).
PRED_LENGTH = "TODO: normalized so với dialect — dài hơn, ngắn hơn hay xấp xỉ?"
PRED_PERPLEXITY = "TODO: PPL(normalized) so với PPL(dialect) — cao hơn, thấp hơn hay xấp xỉ?"
# Nếu gap giữa normalized và standard còn lớn, có nên fine-tune không?
FINETUNE_DECISION_RULE = "TODO: ví dụ - fine-tune nếu PPL(normalized) - PPL(standard) > 0.5"

print({
    "team": TEAM_NAME,
    "rq": NORM_RQ,
    "pred_length": PRED_LENGTH,
    "pred_perplexity": PRED_PERPLEXITY,
    "finetune_rule": FINETUNE_DECISION_RULE,
})

## Tải mô hình baseline

Checkpoint `tarudesu/mbart-large-50` là private. Token **không được hardcode**
trong notebook. Đặt `HF_TOKEN` trong biến môi trường hoặc Colab Secrets:

```python
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
```

Hàm `get_hf_token()` đọc token từ env hoặc Colab Secrets. **Không bao giờ in
token ra output, không commit file `.env`.**

In [ ]:
# Kiểm tra token có sẵn không (không in giá trị token).
token = get_hf_token(required=False)
print("HF_TOKEN available:", token is not None)
if token is not None:
    print("Token length:", len(token), "(không in giá trị)")

In [ ]:
RUN_BASELINE = False  # Đổi True khi đã có GPU và token.
# Dùng subset nhỏ để demo; tăng N_EVAL khi đã kiểm tra thời gian chạy.
N_EVAL = 24  # 8 câu/dialect × 3 dialect

eval_frame = (
    test.sort_values(["target_dialect", "task", "sample_id"])
    .groupby(["target_dialect"], observed=True, group_keys=False)
    .head(N_EVAL // 3)
    .reset_index(drop=True)
)
print("Eval rows:", len(eval_frame))
display(eval_frame[["sample_id", "task", "target_dialect", "dialect_text", "standard_text"]].head())

In [ ]:
output_path = ROOT / "outputs" / "normalization_baseline.csv"
output_path.parent.mkdir(exist_ok=True)

if RUN_BASELINE:
    tokenizer, model, device = load_seq2seq_model(PRIVATE_NORMALIZER_ID, private=True)
    print(f"Loaded {PRIVATE_NORMALIZER_ID} on {device}")
    normalized = generate_normalizations(
        eval_frame["dialect_text"].tolist(),
        tokenizer, model, device,
        max_length=192, batch_size=8,
    )
    baseline = eval_frame.copy()
    baseline["normalized_text"] = normalized
    baseline.to_csv(output_path, index=False)
    print("Saved", output_path)
    # Giải phóng VRAM trước khi tải LM ở dưới.
    import gc, torch
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
elif output_path.exists():
    baseline = pd.read_csv(output_path)
    print("Loaded existing", output_path)
else:
    baseline = pd.DataFrame()
    print("Set RUN_BASELINE=True (cần GPU + HF_TOKEN) để tạo baseline")

## Thí nghiệm trung tâm: so sánh câu gốc và câu dịch

Đây là phần quan trọng nhất của notebook. Với mỗi cặp, ta có **ba phiên bản** của
cùng một ý nghĩa:

1. **`dialect_text`** — câu gốc (input phương ngữ đưa vào normalizer).
2. **`normalized_text`** — câu dịch (output của mBART).
3. **`standard_text`** — bản chuẩn vàng (reference).

Ta so sánh ba phiên bản trên hai tiêu chí:

- **Độ dài** (số từ): bản dịch có dài/ngắn bất thường so với gốc và chuẩn không?
- **Độ phức tạp (perplexity)**: một LM tham chiếu (Qwen2.5-0.5B) gán xác suất
  bao nhiêu cho mỗi phiên bản? PPL thấp hơn = LM "quen" hơn với câu đó.

```text
PPL(text) = exp(NLL(text))
NLL(text) = -(1/(T-1)) * Σ log p(x_t | x_<t)
```

**Câu hỏi then chốt:** bản dịch (normalized) có **gần chuẩn vàng hơn** câu gốc
(dialect) về độ dài và perplexity không? Nếu PPL(normalized) vẫn cao hơn nhiều
PPL(standard), nghĩa là mBART chưa đưa câu về vùng "quen thuộc" của LM → động lực
để fine-tune.

In [ ]:
RUN_PERPLEXITY_COMPARE = False  # Đổi True khi đã có baseline + GPU.

if not baseline.empty:
    # --- Độ dài: số từ của ba phiên bản ---
    length_frame = baseline.assign(
        dialect_words=baseline["dialect_text"].str.split().str.len(),
        normalized_words=baseline["normalized_text"].str.split().str.len(),
        standard_words=baseline["standard_text"].str.split().str.len(),
    )
    length_long = length_frame.melt(
        id_vars=["sample_id", "task", "target_dialect"],
        value_vars=["dialect_words", "normalized_words", "standard_words"],
        var_name="variant", value_name="n_words",
    )

    fig, ax = plt.subplots(figsize=(9, 4.5))
    sns.boxplot(data=length_long, x="target_dialect", y="n_words", hue="variant", ax=ax)
    ax.set(title="Độ dài (số từ): câu gốc vs câu dịch vs chuẩn vàng",
           xlabel="Dialect", ylabel="Số từ")
    plt.tight_layout()
    plt.show()

    length_summary = (
        length_long.groupby(["target_dialect", "variant"], observed=True)
        .n_words.mean().unstack()
        .rename(columns={
            "dialect_words": "dialect(gốc)",
            "normalized_words": "normalized(dịch)",
            "standard_words": "standard(vàng)",
        })
    )
    display(length_summary.round(2))

### Insight (độ dài)

Viết 2–3 câu theo cấu trúc observation → evidence → caveat:

- Bản dịch có dài/ngắn hơn câu gốc không? Có tiệm cận chuẩn vàng không?
- Có khác biệt giữa ba dialect không?
- **Caveat:** độ dài gần nhau không chứng minh nghĩa đã được chuẩn hóa đúng.

In [ ]:
# --- Độ phức tạp (perplexity) của ba phiên bản bằng LM tham chiếu ---
from vialect_seas.probing import load_causal_lm, score_texts

REFERENCE_LM = "Qwen/Qwen2.5-0.5B"  # LM nhỏ, đa ngữ, dùng làm thước "quen thuộc"

ppl_path = ROOT / "outputs" / "normalization_perplexity.csv"

if RUN_PERPLEXITY_COMPARE and not baseline.empty:
    tok, lm, lm_device = load_causal_lm(REFERENCE_LM)
    print(f"Loaded {REFERENCE_LM} on {lm_device}")

    ppl_dialect = score_texts(baseline["dialect_text"], tok, lm, lm_device)
    ppl_normalized = score_texts(baseline["normalized_text"], tok, lm, lm_device)
    ppl_standard = score_texts(baseline["standard_text"], tok, lm, lm_device)

    ppl_frame = baseline[["sample_id", "task", "target_dialect"]].copy()
    ppl_frame["ppl_dialect"] = ppl_dialect["ppl"].values
    ppl_frame["ppl_normalized"] = ppl_normalized["ppl"].values
    ppl_frame["ppl_standard"] = ppl_standard["ppl"].values
    ppl_frame["nll_dialect"] = ppl_dialect["nll"].values
    ppl_frame["nll_normalized"] = ppl_normalized["nll"].values
    ppl_frame["nll_standard"] = ppl_standard["nll"].values
    ppl_frame.to_csv(ppl_path, index=False)
    print("Saved", ppl_path)

    import gc, torch
    del lm, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
elif ppl_path.exists():
    ppl_frame = pd.read_csv(ppl_path)
    print("Loaded existing", ppl_path)
else:
    ppl_frame = pd.DataFrame()
    print("Set RUN_PERPLEXITY_COMPARE=True (cần baseline + GPU) để tính perplexity")

In [ ]:
if not ppl_frame.empty:
    ppl_long = ppl_frame.melt(
        id_vars=["sample_id", "task", "target_dialect"],
        value_vars=["ppl_dialect", "ppl_normalized", "ppl_standard"],
        var_name="variant", value_name="perplexity",
    )
    # Cắt giá trị cực đại để biểu đồ không bị một điểm outliers kéo giật.
    ppl_long_capped = ppl_long.copy()
    upper = ppl_long["perplexity"].quantile(0.95)
    ppl_long_capped["perplexity"] = ppl_long_capped["perplexity"].clip(upper=upper)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.boxplot(data=ppl_long_capped, x="target_dialect", y="perplexity",
                hue="variant", ax=axes[0])
    axes[0].set(title="Perplexity: câu gốc vs câu dịch vs chuẩn vàng",
                xlabel="Dialect", ylabel="Perplexity (capped 95%)")

    # Delta perplexity so với chuẩn vàng.
    ppl_frame["delta_ppl_norm_vs_std"] = ppl_frame["ppl_normalized"] - ppl_frame["ppl_standard"]
    ppl_frame["delta_ppl_dia_vs_std"] = ppl_frame["ppl_dialect"] - ppl_frame["ppl_standard"]
    delta_long = ppl_frame.melt(
        id_vars=["sample_id", "task", "target_dialect"],
        value_vars=["delta_ppl_dia_vs_std", "delta_ppl_norm_vs_std"],
        var_name="comparison", value_name="delta_ppl",
    )
    delta_capped = delta_long.copy()
    delta_capped["delta_ppl"] = delta_capped["delta_ppl"].clip(
        lower=delta_long["delta_ppl"].quantile(0.05),
        upper=delta_long["delta_ppl"].quantile(0.95),
    )
    sns.boxplot(data=delta_capped, x="target_dialect", y="delta_ppl",
                hue="comparison", ax=axes[1])
    axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[1].set(title="Delta PPL so với chuẩn vàng (<0 = tốt hơn chuẩn)",
                xlabel="Dialect", ylabel="PPL - PPL(standard)")
    plt.tight_layout()
    plt.show()

    ppl_summary = ppl_frame.groupby("target_dialect", observed=True).agg(
        ppl_dialect=("ppl_dialect", "mean"),
        ppl_normalized=("ppl_normalized", "mean"),
        ppl_standard=("ppl_standard", "mean"),
        gap_norm_vs_std=("delta_ppl_norm_vs_std", "mean"),
        gap_dia_vs_std=("delta_ppl_dia_vs_std", "mean"),
    )
    display(ppl_summary.round(3))

### Insight (perplexity) — phần bắt buộc

Trả lời các câu sau bằng số từ biểu đồ:

1. **PPL(dialect) vs PPL(standard):** câu gốc phương ngữ có perplexity cao hơn
   chuẩn vàng không? (khoảng cách bao nhiêu?)
2. **PPL(normalized) vs PPL(standard):** bản dịch có tiệm cận chuẩn vàng không?
   khoảng cách còn lại bao nhiêu?
3. **PPL(normalized) vs PPL(dialect):** normalization có giảm perplexity so với
   câu gốc không?
4. **Quyết định fine-tune:** theo rule nhóm đặt ở STUDENT TASK 0, gap còn lại có
   đủ lớn để fine-tune không?

**Caveat:** PPL đo "quen thuộc" với LM tham chiếu, không phải đúng/sai semantic.
Một câu có thể PPL thấp nhưng sai nghĩa. Luôn kết hợp với CER/WER ở phần sau.

## Đánh giá bản dịch bằng metric dịch máy

Ngoài độ dài và perplexity, đo khoảng cách giữa bản dịch và chuẩn vàng bằng:

- **CER (Character Error Rate)** = edit distance / ký tự reference.
- **WER (Word Error Rate)** = edit distance / từ reference.
- **Exact match** = 1 nếu bản dịch == chuẩn vàng (sau khi strip).

CER thấp = bản dịch giống chuẩn vàng về bề mặt. Nhưng CER không thưởng nghĩa;
dùng kèm kiểm tra thủ công.

In [ ]:
if not baseline.empty:
    baseline = baseline.assign(
        cer=[character_error_rate(ref, hyp)
             for ref, hyp in zip(baseline["standard_text"], baseline["normalized_text"])],
        wer=[word_error_rate(ref, hyp)
             for ref, hyp in zip(baseline["standard_text"], baseline["normalized_text"])],
        exact_match=[exact_match(ref, hyp)
                     for ref, hyp in zip(baseline["standard_text"], baseline["normalized_text"])],
    )
    metric_by_dialect = (
        baseline.groupby("target_dialect", observed=True)
        .agg(mean_cer=("cer", "mean"),
             mean_wer=("wer", "mean"),
             exact_match_rate=("exact_match", "mean"))
        .reset_index()
    )
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=metric_by_dialect, x="target_dialect", y="mean_cer", ax=ax)
    ax.set(title="CER giữa bản dịch mBART và chuẩn vàng",
           xlabel="Dialect", ylabel="Character Error Rate")
    plt.tight_layout()
    plt.show()
    display(metric_by_dialect.round(3))

### STUDENT TASK 1 — Phân tích lỗi thủ công

Metric chỉ cho số. Nhóm phải đọc bản dịch và phân loại lỗi. Chọn ít nhất 10 câu
có CER cao nhất và gán nhãn lỗi theo taxonomy: `lexical` (sai từ vựng),
`morphology` (hậu tố/chia thì), `syntax` (trật tự từ), `discourse` (mạch lạc),
`meaning` (sai nghĩa), `fluent` (không lỗi).

In [ ]:
RUN_ERROR_ANALYSIS = False

if RUN_ERROR_ANALYSIS and not baseline.empty:
    # HINT 1: sort theo CER giảm dần, lấy top 10-15 câu.
    # HINT 2: với mỗi câu, in dialect_text / normalized_text / standard_text cạnh nhau.
    # HINT 3: gán nhãn lỗi vào cột `error_type`; đếm tần suất theo error_type.
    """Your code here"""
    top_errors = None  # thay bằng DataFrame của nhóm

    # SELF-CHECK
    assert top_errors is not None, "Hãy tạo biến top_errors"
    assert "error_type" in top_errors.columns
    assert len(top_errors) >= 10
    display(top_errors[["sample_id", "target_dialect", "dialect_text",
                        "normalized_text", "standard_text", "cer", "error_type"]].head(12))

## Fine-tune bằng LoRA

Nếu gap perplexity giữa bản dịch và chuẩn vàng còn lớn (theo rule ở TASK 0),
ta fine-tune mBART. Nhưng mBART-large có hàng trăm triệu tham số; train full
tốn VRAM lớn. **LoRA** (Low-Rank Adaptation) thêm ma trận thấp hạng vào weight
có sẵn, chỉ train phần nhỏ đó.

```text
W' = W + (alpha / r) * B @ A
```

- `W` ∈ R^(d×d): weight gốc, đóng băng.
- `A` ∈ R^(r×d), `B` ∈ R^(d×r): hai ma trận thấp hạng, rank `r` nhỏ (ví dụ 8).
- `alpha`: scaling.
- Chỉ `A`, `B` được update; tham số gốc giữ nguyên → tiết kiệm VRAM.

Tham khảo CS224n: đây là một dạng **parameter-efficient fine-tuning**. Trong đồ
án, LoRA gắn vào `q_proj` và `v_proj` của attention (các projection hay nhạy với
tác vụ).

### Cài đặt training arguments

Trừ `fp16` và `predict_with_generate` ra, thì những giá trị còn lại các em có nên
thay đổi không? Điền cột **Giải thích ý nghĩa** và điều chỉnh **Giá trị Gợi ý**
nếu thấy cần (ghi rõ lý do vào báo cáo).

| Argument | Giá trị Gợi ý | Giải thích ý nghĩa |
| --- | --- | --- |
| `output_dir` | `"outputs/lora_run"` | VD: đường dẫn thư mục lưu kết quả training |
| `num_train_epochs` | `3` | |
| `per_device_train_batch_size` | `4` | |
| `per_device_eval_batch_size` | `8` | |
| `learning_rate` | `2e-4` | |
| `warmup_ratio` | `0.05` | |
| `weight_decay` | `0.01` | |
| `save_strategy` | `"no"` | |
| `logging_steps` | `10` | |
| `report_to` | `[]` | |
| `fp16` | **True** (nếu có GPU) | |
| `predict_with_generate` | **False** | |
| `gradient_accumulation_steps` | `2` | |

In [ ]:
RUN_FINETUNE = False  # Đổi True khi đã có GPU >= T4 và đã chạy baseline.
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# Cell này chỉ chạy khi RUN_FINETUNE = True. Điền các chỗ ______ trước khi chạy.
if RUN_FINETUNE:
    import torch
    from datasets import Dataset
    from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer,
                              DataCollatorForSeq2Seq, Seq2SeqTrainer,
                              Seq2SeqTrainingArguments)

    # CODE: Lấy HF token (không hardcode) qua get_hf_token(required=False)
    token = get_hf_token(required=False)
    kwargs = {"token": token} if token else {}

    # CODE: Tải tokenizer và base model từ BASE_MODEL_ID
    tokenizer = ______
    base_model = ______

    # CODE: Đặt src_lang và tgt_lang = "vi_VN" cho mBART (nếu tokenizer hỗ trợ)
    # HINT: dùng hasattr(tokenizer, "src_lang") để kiểm tra trước khi gán.
    if hasattr(tokenizer, "src_lang"):
        tokenizer.src_lang = ______
    if hasattr(tokenizer, "tgt_lang"):
        tokenizer.tgt_lang = ______

    # CODE: Gắn LoRA vào base_model (q_proj, v_proj) bằng attach_lora()
    # HINT: truyền rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT.
    model = attach_lora(______, rank=______, alpha=______, dropout=______)
    model.print_trainable_parameters()

    # CODE: Tạo Dataset từ train (không dùng test) và tokenize bằng make_preprocess_function
    # HINT: Dataset.from_pandas(train.reset_index(drop=True)); map(preprocess, batched=True, remove_columns=...)
    train_ds = ______
    preprocess = make_preprocess_function(tokenizer, max_length=192)
    tokenized = ______

    # CODE: Tạo DataCollatorForSeq2Seq (padding, label_pad_token_id=-100)
    data_collator = ______

    # CODE: Thiết lập Seq2SeqTrainingArguments với các giá trị ở bảng trên
    # HINT: dùng fp16=torch.cuda.is_available() và predict_with_generate=False.
    args = Seq2SeqTrainingArguments(
        output_dir=______,
        num_train_epochs=______,
        per_device_train_batch_size=______,
        per_device_eval_batch_size=______,
        learning_rate=______,
        save_strategy=______,
        logging_steps=______,
        report_to=______,
        fp16=______,
        predict_with_generate=______,
    )

    # CODE: Tạo Seq2SeqTrainer và bắt đầu training
    # HINT: truyền model, args, train_dataset, tokenizer, data_collator.
    trainer = Seq2SeqTrainer(
        model=______,
        args=______,
        train_dataset=______,
        tokenizer=______,
        data_collator=______,
    )
    trainer.train()

    # CODE: Lưu LoRA adapter vào outputs/lora_adapter
    # HINT: model.save_pretrained(...) và tokenizer.save_pretrained(...)
    ______
    print("Saved LoRA adapter to outputs/lora_adapter")
else:
    print("Set RUN_FINETUNE=True (cần GPU) để fine-tune. Đọc code và điền ______ trước khi chạy.")

### STUDENT TASK 2 — Đánh giá fine-tuned vs baseline

Tải lại base model, gắn LoRA adapter đã train, sinh bản dịch trên cùng eval_frame
ở trên, rồi so sánh CER/WER/perplexity với baseline. Bảng so sánh phải dùng cùng
test split và cùng metric.

In [ ]:
RUN_EVAL_FINETUNED = False

def load_finetuned_and_generate(eval_df):
    # HINT 1: AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_ID) rồi PeftModel.from_pretrained(model, adapter_path).
    # HINT 2: Dùng generate_normalizations() với model đã gắn adapter.
    # HINT 3: Trả về DataFrame có thêm cột normalized_finetuned, cer_finetuned, wer_finetuned.
    """Your code here"""
    return None

if RUN_EVAL_FINETUNED:
    comparison = load_finetuned_and_generate(eval_frame)
    if comparison is None:
        raise NotImplementedError("Hoàn thành load_finetuned_and_generate trước khi bật cờ")
    # SELF-CHECK: phải có cả cột baseline và finetuned để so sánh công bằng.
    required_cols = {"normalized_text", "normalized_finetuned",
                     "cer", "cer_finetuned", "wer", "wer_finetuned"}
    assert required_cols.issubset(comparison.columns)
    assert len(comparison) == len(eval_frame), "Phải đánh giá trên cùng eval_frame"
    display(comparison[["sample_id", "target_dialect", "cer", "cer_finetuned"]].head())

### Insight (fine-tune)

Trả lời:

1. CER/WER có giảm sau fine-tune không? Giảm bao nhiêu (trung bình + theo dialect)?
2. Perplexity của bản dịch fine-tuned có tiệm cận chuẩn vàng hơn không?
3. Có dialect nào được lợi nhiều hơn từ fine-tune không? Vì sao?
4. **Caveat:** train_240 rất nhỏ (240 cặp). Overfitting có thể làm CER giảm trên
   eval mẫu nhưng không khái quát. Báo cáo cả eval trong-sample và out-of-sample.

**Quy tắc công bằng:** luôn giữ cùng test split, cùng metric, cùng decoding config
(`do_sample=False, num_beams=1`) khi so baseline vs fine-tuned.

## IV. Demo

Phần demo cho thuyết trình: chọn một câu dialect, chạy baseline và model đã
fine-tune, in kết quả cạnh nhau và phân tích. Mỗi cell dưới đây là bài tập của
nhóm — điền code vào marker `Your code here`.

In [ ]:
# Chọn 1 mẫu dialect từ test set và in câu gốc + chuẩn vàng
# HINT: dùng eval_frame.iloc[[idx]] hoặc sample ngẫu nhiên với seed cố định.

"""Your code here"""

In [ ]:
# Load model fine-tuned (base + LoRA adapter) và sinh bản dịch cho mẫu đã chọn
# HINT 1: AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_ID) rồi PeftModel.from_pretrained(model, "outputs/lora_adapter").
# HINT 2: Dùng generate_normalizations() với model đã gắn adapter.

"""Your code here"""

In [ ]:
# In bản dialect / bản dịch baseline / bản dịch fine-tuned / chuẩn vàng cạnh nhau,
# tính CER của từng bản dịch và viết 1-2 câu nhận xét.

"""Your code here"""

## Bài tập mở

1. Thay đổi `LORA_RANK` (4, 8, 16) và đo trade-off VRAM vs CER.
2. Thử target modules khác (`k_proj`, `o_proj`) và giải thích vì sao.
3. So sánh decoding: `num_beams=1` (greedy) vs `num_beams=4` (beam search).
4. Tính correlation giữa CER và delta-PPL trên tập eval.
5. Kiểm tra: có câu nào CER = 0 nhưng PPL(normalized) >> PPL(standard) không?
   Tại sao (gợi ý: synonym, ngữ pháp đúng nhưng hiếm)?
6. Viết kết luận: normalization có giảm gap phương ngữ không? Fine-tune có giúp
   thêm không? Hạn chế và hướng tiếp theo.

In [ ]:
# STUDENT TASK 3 (EXTENSION) — Một thí nghiệm mở do nhóm thiết kế.
RUN_STUDENT_EXPERIMENT = False

def student_normalization_experiment(baseline_df, ppl_df):
    # HINT 1: chọn một biến chưa thử (rank, target module, decoding, eval subset).
    # HINT 2: giữ một baseline so sánh công bằng trên cùng split + metric.
    # HINT 3: trả về dict có "summary" (DataFrame) và "figure" (matplotlib Figure).
    """Your code here"""
    return None

if RUN_STUDENT_EXPERIMENT:
    result = student_normalization_experiment(baseline, ppl_frame)
    if result is None:
        raise NotImplementedError("Hoàn thành student_normalization_experiment trước khi bật cờ")
    assert isinstance(result, dict)
    assert {"summary", "figure"}.issubset(result)
    display(result["summary"])
    plt.show()

In [ ]:
# Lưu artifact nộp bài.
output_dir = ROOT / "outputs"
output_dir.mkdir(exist_ok=True)
if not baseline.empty:
    baseline.to_csv(output_dir / "team_normalization_baseline.csv", index=False)
    print("Saved", output_dir / "team_normalization_baseline.csv")
if not ppl_frame.empty:
    ppl_frame.to_csv(output_dir / "team_normalization_perplexity.csv", index=False)
    print("Saved", output_dir / "team_normalization_perplexity.csv")